# Exercice technique Verbus

## Chargement des données

In [4]:
import pandas as pd
import json
import time

df_history = pd.read_csv('data/bornes_history.csv')
df_history["timestamp"] = pd.to_datetime(df_history["timestamp"], format='ISO8601', utc=True)
df_planning = pd.read_csv('data/planning.csv', parse_dates =["arrival_time", "departure_time"])
df_sessions = pd.read_csv('data/sessions.csv', parse_dates=["start_time", "end_time"])

with open('data/bornes.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df_bornes = pd.DataFrame(data)
df_bornes["last_update"] = pd.to_datetime(df_bornes["last_update"], format='ISO8601', utc=True)

## Exploration initiale des données

In [2]:
df_bornes.info()
df_bornes.head()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   station_id    6 non-null      str                
 1   connector_id  6 non-null      int64              
 2   status        6 non-null      str                
 3   power_kw      6 non-null      int64              
 4   last_update   6 non-null      datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), int64(2), str(2)
memory usage: 372.0 bytes


,station_id,connector_id,status,power_kw,last_update
0,STAT-01,1,Available,11,2025-04-13 19:00:00+00:00
1,STAT-01,2,Available,11,2025-04-08 04:00:00+00:00
2,STAT-02,1,Available,11,2025-04-07 06:00:00+00:00
3,STAT-02,2,Faulted,11,2025-04-09 08:00:00+00:00
4,STAT-03,1,Unavailable,11,2025-04-09 22:00:00+00:00


In [3]:
df_bornes.info()
df_bornes.head()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   station_id    6 non-null      str                
 1   connector_id  6 non-null      int64              
 2   status        6 non-null      str                
 3   power_kw      6 non-null      int64              
 4   last_update   6 non-null      datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), int64(2), str(2)
memory usage: 372.0 bytes


,station_id,connector_id,status,power_kw,last_update
0,STAT-01,1,Available,11,2025-04-13 19:00:00+00:00
1,STAT-01,2,Available,11,2025-04-08 04:00:00+00:00
2,STAT-02,1,Available,11,2025-04-07 06:00:00+00:00
3,STAT-02,2,Faulted,11,2025-04-09 08:00:00+00:00
4,STAT-03,1,Unavailable,11,2025-04-09 22:00:00+00:00


In [4]:
df_planning.info()

<class 'pandas.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   vehicle_id           172 non-null    str                
 1   vehicle_type         172 non-null    str                
 2   arrival_time         172 non-null    datetime64[us, UTC]
 3   departure_time       172 non-null    datetime64[us, UTC]
 4   service_km           172 non-null    float64            
 5   service_hours        172 non-null    float64            
 6   required_charge_kwh  172 non-null    float64            
dtypes: datetime64[us, UTC](2), float64(3), str(2)
memory usage: 9.5 KB


In [5]:
df_planning.info()

<class 'pandas.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   vehicle_id           172 non-null    str                
 1   vehicle_type         172 non-null    str                
 2   arrival_time         172 non-null    datetime64[us, UTC]
 3   departure_time       172 non-null    datetime64[us, UTC]
 4   service_km           172 non-null    float64            
 5   service_hours        172 non-null    float64            
 6   required_charge_kwh  172 non-null    float64            
dtypes: datetime64[us, UTC](2), float64(3), str(2)
memory usage: 9.5 KB


In [6]:
df_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   station_id    111 non-null    str                
 1   connector_id  111 non-null    int64              
 2   timestamp     111 non-null    datetime64[us, UTC]
 3   status        111 non-null    str                
dtypes: datetime64[us, UTC](1), int64(1), str(2)
memory usage: 3.6 KB


In [7]:
df_sessions.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   session_id          52 non-null     str                
 1   station_id          52 non-null     str                
 2   connector_id        52 non-null     int64              
 3   vehicle_id          52 non-null     str                
 4   start_time          52 non-null     datetime64[us, UTC]
 5   end_time            51 non-null     datetime64[us, UTC]
 6   energy_kwh          52 non-null     float64            
 7   effective_power_kw  52 non-null     int64              
dtypes: datetime64[us, UTC](2), float64(1), int64(2), str(3)
memory usage: 3.4 KB


## Compréhension des fichiers et des liens entre eux

bornes.json décrit l’infrastructure de recharge “à l’instant T” : pour chaque station et connecteur, on connaît le statut courant (Available, Charging, Faulted, Unavailable), la puissance nominale (11 kW) et la date de dernière mise à jour.

sessions.csv retrace, sur la période étudiée, les sessions de recharge réalisées : chaque ligne correspond à une session sur un connecteur donné, pour un véhicule donné, avec les horodatages de début/fin, l’énergie délivrée et la puissance effective de charge.

planning.csv décrit le planning d’exploitation de la flotte : pour chaque véhicule et chaque sortie, on dispose des horaires d’arrivée/départ au dépôt, des kilomètres parcourus, de la durée de service et de l’énergie estimée nécessaire pour le prochain service.

bornes_history.csv détaille l’historique des changements d’état des bornes : pour chaque station et connecteur, on suit dans le temps les passages entre Available, Charging, Faulted et Unavailable.

Ensemble, ces quatre fichiers permettent de relier le besoin opérationnel (planning des véhicules et énergie requise) à l’usage réel des bornes (sessions de recharge) et à la disponibilité effective de l’infrastructure dans le temps (historique des statuts).

## Vérifications de cohérence

In [8]:
print("=== Vérifications de cohérence - bornes.json ===")

print("Nombre de connecteurs :", len(df_bornes))
print("Nombre de stations :", df_bornes["station_id"].nunique())

print("\nStatuts observés :")
print(df_bornes["status"].value_counts())

print("\nPuissances nominales distinctes (kW) :")
print(df_bornes["power_kw"].unique())

=== Vérifications de cohérence - bornes.json ===
Nombre de connecteurs : 6
Nombre de stations : 3

Statuts observés :
status
Available      4
Faulted        1
Unavailable    1
Name: count, dtype: int64

Puissances nominales distinctes (kW) :
[11]


In [9]:
print("=== Vérifications de cohérence - sessions.csv ===")

print("Nombre de sessions :", len(df_sessions))
print("Nombre de véhicules distincts :", df_sessions["vehicle_id"].nunique())

print("\nSessions incomplètes :")
print(df_sessions["end_time"].isna().sum())

print("\nStatistiques énergie (kWh) :")
print(df_sessions["energy_kwh"].describe())

=== Vérifications de cohérence - sessions.csv ===
Nombre de sessions : 52
Nombre de véhicules distincts : 18

Sessions incomplètes :
1

Statistiques énergie (kWh) :
count    52.000000
mean     27.397115
std      13.988860
min       0.000000
25%      19.540000
50%      24.190000
75%      35.625000
max      62.170000
Name: energy_kwh, dtype: float64


In [10]:
print("=== Vérifications de cohérence - planning.csv ===")

print("Nombre de lignes de planning :", len(df_planning))
print("Nombre de véhicules distincts :", df_planning["vehicle_id"].nunique())

print("\nRépartition des types de véhicules :")
print(df_planning["vehicle_type"].value_counts())

print("\nStatistiques service_km :")
print(df_planning["service_km"].describe())

print("\nStatistiques service_hours :")
print(df_planning["service_hours"].describe())

=== Vérifications de cohérence - planning.csv ===
Nombre de lignes de planning : 172
Nombre de véhicules distincts : 20

Répartition des types de véhicules :
vehicle_type
Renault Zoé      70
Citroën Jumpy    53
Kia EV3          49
Name: count, dtype: int64

Statistiques service_km :
count    172.000000
mean     148.786628
std       38.420450
min       81.300000
25%      121.250000
50%      149.950000
75%      172.900000
max      248.700000
Name: service_km, dtype: float64

Statistiques service_hours :
count    172.000000
mean       6.127326
std        1.636680
min        3.100000
25%        4.975000
50%        6.200000
75%        7.500000
max        9.000000
Name: service_hours, dtype: float64


In [11]:
print("=== Vérifications de cohérence - bornes_history.csv ===")

print("Nombre total d'événements :", len(df_history))

print("\nStations couvertes :")
print(df_history["station_id"].unique())

print("Nombre de connecteurs distincts :",
      df_history[["station_id", "connector_id"]].drop_duplicates().shape[0])

print("\nStatuts observés :")
print(df_history["status"].value_counts())

print("\nPériode couverte :")
print("Min :", df_history["timestamp"].min())
print("Max :", df_history["timestamp"].max())

=== Vérifications de cohérence - bornes_history.csv ===
Nombre total d'événements : 111

Stations couvertes :
<StringArray>
['STAT-01', 'STAT-02', 'STAT-03']
Length: 3, dtype: str
Nombre de connecteurs distincts : 6

Statuts observés :
status
Available      57
Charging       52
Faulted         1
Unavailable     1
Name: count, dtype: int64

Période couverte :
Min : 2025-04-07 00:00:00+00:00
Max : 2025-04-11 20:00:00+00:00


## KPI 1 : Utilisation des bornes

In [5]:
print("=== KPI 1 - Utilisation des bornes (sessions valides) ===")

sessions_valides = df_sessions[
    (df_sessions["end_time"].notna()) &
    (df_sessions["energy_kwh"] >= 1)
]

kpi_bornes = (
    sessions_valides
    .groupby(["station_id", "connector_id"])
    .agg(
        nb_sessions=("session_id", "size"),
        energie_totale_kwh=("energy_kwh", "sum"),
        energie_moyenne_kwh=("energy_kwh", "mean")
    )
    .reset_index()
    .sort_values(["station_id", "connector_id"])
)

display(kpi_bornes)

print(
    "\nÉnergie totale délivrée sur la période (sessions valides, kWh) :",
    sessions_valides["energy_kwh"].sum()
)

=== KPI 1 - Utilisation des bornes (sessions valides) ===


,station_id,connector_id,nb_sessions,energie_totale_kwh,energie_moyenne_kwh
0,STAT-01,1,8,257.67,32.208750
1,STAT-01,2,11,272.98,24.816364
2,STAT-02,1,9,314.84,34.982222
3,STAT-02,2,3,94.66,31.553333
4,STAT-03,1,10,223.42,22.342000
5,STAT-03,2,7,259.44,37.062857



Énergie totale délivrée sur la période (sessions valides, kWh) : 1423.01


Pour l’analyse de l’utilisation des bornes, je me limite aux sessions “valides”, c’est‑à‑dire celles qui ont une fin (end_time renseigné) et au moins 1 kWh délivré ; les sessions incomplètes ou quasi nulles sont analysées séparément comme anomalies.

Sur cette base, les 6 connecteurs délivrent au total 1 423,01 kWh, avec entre 3 et 11 sessions valides par connecteur.

Les connecteurs les plus sollicités sont STAT‑01‑2 (11 sessions, 272,98 kWh) et STAT‑03‑1 (10 sessions, 223,42 kWh), tandis que STAT‑02‑2 n’est utilisé que 3 fois (94,66 kWh).

L’énergie moyenne par session valide varie d’environ 22 kWh (STAT‑03‑1) à 37 kWh (STAT‑03‑2), ce qui correspond à des compléments de charge significatifs plutôt qu’à des recharges complètes.

## KPI 2 : Indisponibilite des bornes

In [18]:

df_h = df_history.sort_values(["station_id", "connector_id", "timestamp"]).copy()
df_h["next_timestamp"] = df_h.groupby(["station_id", "connector_id"])["timestamp"].shift(-1)

fin_periode = df_h["timestamp"].max().ceil("D")
df_h["next_timestamp"] = df_h["next_timestamp"].fillna(fin_periode)
df_h["duration_h"] = (df_h["next_timestamp"] - df_h["timestamp"]).dt.total_seconds() / 3600

df_h["status"] = df_h["status"].str.strip().str.lower()

duree_par_statut = (
    df_h.groupby(["station_id", "connector_id", "status"], as_index=False)["duration_h"]
    .sum()
    .pivot_table(
        index=["station_id", "connector_id"],
        columns="status",
        values="duration_h",
        fill_value=0
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

colonnes_statuts = [c for c in duree_par_statut.columns if c not in ["station_id", "connector_id"]]
duree_par_statut["total_h"] = duree_par_statut[colonnes_statuts].sum(axis=1)

for col in colonnes_statuts:
    duree_par_statut[f"pct_{col}"] = 100 * duree_par_statut[col] / duree_par_statut["total_h"]

colonnes_pct = ["station_id", "connector_id"] + [f"pct_{c}" for c in colonnes_statuts]
display(duree_par_statut[colonnes_pct].sort_values(["station_id", "connector_id"]))

,station_id,connector_id,pct_available,pct_charging,pct_faulted,pct_unavailable
0,STAT-01,1,82.633333,17.366667,0.000000,0.000000
1,STAT-01,2,75.256399,24.743601,0.000000,0.000000
2,STAT-02,1,75.499362,24.500638,0.000000,0.000000
3,STAT-02,2,39.545455,7.121212,53.333333,0.000000
4,STAT-03,1,82.069697,14.596970,0.000000,3.333333
5,STAT-03,2,78.888063,21.111937,0.000000,0.000000


Ce code sert à mesurer le pourcentage du temps où les bornes sont en panne ou hors service.

je reconstruis la durée de chaque état de chaque borne, puis je cumule toutes les périodes où le statut est Faulted ou Unavailable et je compare ce temps au temps total pour obtenir le taux d’indisponibilité.

## KPI 3 : Recharge des vehicules et couverture des besoins

In [3]:

besoin_par_vehicule = (
    df_planning
    .groupby("vehicle_id", as_index=False)["required_charge_kwh"]
    .sum()
)


recharge_par_vehicule = (
    df_sessions
    .groupby("vehicle_id", as_index=False)["energy_kwh"]
    .sum()
)


df_kpi3 = besoin_par_vehicule.merge(
    recharge_par_vehicule,
    on="vehicle_id",
    how="left"
)


df_kpi3["energy_kwh"] = df_kpi3["energy_kwh"].fillna(0)


df_kpi3["taux_couverture"] = (
    df_kpi3["energy_kwh"] / df_kpi3["required_charge_kwh"]
)


besoin_total = df_kpi3["required_charge_kwh"].sum()
recharge_totale = df_kpi3["energy_kwh"].sum()

taux_couverture_global = (
    recharge_totale / besoin_total if besoin_total > 0 else 0
)

display(df_kpi3)
print(f"Taux global de couverture des besoins de recharge : {taux_couverture_global:.1%}")

,vehicle_id,required_charge_kwh,energy_kwh,taux_couverture
0,VEH-001,196.86,56.60,0.287514
1,VEH-002,219.24,93.85,0.428070
2,VEH-003,232.98,58.24,0.249979
3,VEH-004,130.52,102.14,0.782562
4,VEH-005,225.62,53.01,0.234953
5,VEH-006,186.95,0.00,0.000000
6,VEH-007,209.94,48.15,0.229351
7,VEH-008,184.08,38.34,0.208279
8,VEH-009,250.79,99.14,0.395311
9,VEH-010,240.40,51.13,0.212687


Taux global de couverture des besoins de recharge : 26.1%


Sur ce KPI, on voit que les véhicules sont globalement moins rechargés que ce qui était prévu dans le planning.

En consolidant les 20 véhicules du tableau, le besoin total atteint 5 457,24 kWh alors que la recharge observée s’élève à 1 424,65 kWh, soit un taux global de couverture de 26,1%.

Les écarts restent marqués selon les véhicules : VEH‑001 couvre 28,8% de ses besoins (56,60 kWh sur 196,86 kWh), VEH‑003 atteint 25,0% (58,24 kWh sur 232,98 kWh) et VEH‑004 s’en sort mieux avec 78,3% de couverture (102,14 kWh sur 130,52 kWh). À l’autre extrême, VEH‑006 et VEH‑015 ne reçoivent aucune recharge sur la période.

La lecture complète du tableau confirme donc une sous‑recharge généralisée de la flotte, avec seulement quelques véhicules proches d’un niveau de couverture satisfaisant.

## KPI 4 : Creneaux de tension

In [4]:
evenements = []

for _, ligne in df_sessions.iterrows():
    evenements.append({
        "instant": ligne["start_time"],
        "variation": 1
    })
    evenements.append({
        "instant": ligne["end_time"],
        "variation": -1
    })

evenements_df = pd.DataFrame(evenements)

evenements_df = evenements_df.sort_values("instant")

evenements_df["nb_vehicules_en_charge"] = evenements_df["variation"].cumsum()

creux_tension = evenements_df[evenements_df["nb_vehicules_en_charge"] >= 5]

creux_tension.head(20)

Nombre de créneaux de tension (>= 5 véhicules) : 8
Pic de véhicules en charge simultanée : 6


,debut,fin,duree_h,pic_vehicules
0,2025-04-07 14:10:16.927315+00:00,2025-04-07 14:20:28.457722+00:00,0.169870,5
1,2025-04-08 13:25:07.637138+00:00,2025-04-08 13:30:25.233107+00:00,0.088221,5
2,2025-04-08 13:35:59.162823+00:00,2025-04-08 14:36:46.993718+00:00,1.013286,5
3,2025-04-08 14:52:40.124603+00:00,2025-04-08 14:59:26.032407+00:00,0.112752,5
4,2025-04-09 13:02:17.997184+00:00,2025-04-09 15:28:37.930113+00:00,2.438870,6
5,2025-04-09 15:33:29.162171+00:00,2025-04-09 17:10:03.800682+00:00,1.609622,6
6,2025-04-10 13:35:22.192073+00:00,2025-04-10 15:11:33.164570+00:00,1.603048,6
7,2025-04-10 22:34:49.183070+00:00,2025-04-11 00:17:22.026497+00:00,1.709123,5


Pour ce KPI, je calcule combien de véhicules sont en charge en même temps, en comptant +1 à chaque début de session et −1 à chaque fin. Dès que ce nombre atteint au moins 5 véhicules, je considère que l’on est dans un créneau de tension. 

## Visualisations

sessions.csv : certaines sessions sont très longues avec moins de 1 kWh délivré, ce qui semble peu réaliste pour une vraie recharge. Je les filtre (end_time non nul et energy_kwh ≥ 1) pour ne pas biaiser les moyennes. De plus, la puissance effective affichée est parfois proche de 22 kW alors que le contexte évoque des bornes lentes AC, ce qui crée une ambiguïté sur la puissance réelle des points de charge. Par manque d’informations complémentaires, j’utilise surtout l’énergie délivrée (energy_kwh) comme base pour mes KPI.

planning.csv : le besoin required_charge_kwh est pris comme référence des besoins de la flotte, mais je ne dispose pas du détail de son calcul ni du niveau de charge initial des véhicules à l’arrivée ni de la capacité batterie. Je peux donc mesurer un taux de couverture en kWh, mais pas raisonner en pourcentage de batterie ou en marge de sécurité.

bornes_history.csv : l’historique donne les changements d’état des bornes mais pas la fin explicite du dernier état pour chaque borne. Pour calculer des durées par état, je suppose que ce dernier état reste valable jusqu’à la fin de la période observée, ce qui peut légèrement sur‑estimer la durée du dernier état. Par ailleurs, les états Faulted et Unavailable sont comptés comme indisponibles sans détail sur la cause (panne vs maintenance), ce qui limite l’analyse fine des origines d’indisponibilité.

Enfin, aucune des sources ne donne la puissance maximale disponible au niveau du dépôt ni les éventuelles règles de pilotage de la charge, ce qui m’empêche de distinguer clairement ce qui relève d’une limite d’infrastructure locale (bornes) d’une limite de réseau ou de stratégie de gestion de la charge.